# Hidden Object Detection (Cluttered Scenes)

This notebook simulates cluttered scenes with hidden metallic objects, applies denoising/background subtraction, and runs the trained classifier on those samples to evaluate robustness.

In [ ]:
import os
import numpy as np
from radar_simulation import generate_range_doppler_heatmap, denoise_background_subtract, visualize_heatmap
from tensorflow.keras.models import load_model

models_dir = os.path.join('.', 'models')
model_path = os.path.join(models_dir, 'metal_classifier.h5')
if not os.path.exists(model_path):
    raise FileNotFoundError('Model not found. Run classification_model.ipynb first to train and save the model.')
model = load_model(model_path)

# Generate test hidden samples
tests = []
for i in range(20):
    hm = generate_range_doppler_heatmap(64, 64, scenario='hidden', metal=True, clutter_level=0.25, snr_db=8)
    proc = denoise_background_subtract(hm, method='median', kernel_size=5)
    tests.append((hm, proc))

# Predict and show some examples
for idx, (raw, proc) in enumerate(tests[:8]):
    x = proc[None, ..., None].astype('float32')
    p = model.predict(x)[0,0]
    print(f'idx={idx} pred_prob={p:.3f} -> class={'metal' if p>=0.5 else 'non-metal'}')
    visualize_heatmap(raw, title=f'raw idx={idx}', show=True)
    visualize_heatmap(proc, title=f'proc idx={idx} pred_prob={p:.3f}', show=True)